# SkyQuery E2: held-out classes
Retrains YOLO26s with **bus, bicycle and awning-tricycle** removed from the labels (same recipe as the original model: 640 px tiles + full frames, 40 epochs, seed 42), then asks how well a zero-shot detector covers the classes the trained model has never seen. About 7 hours on 2×T4; the training cell resumes if the session restarts.

In [ ]:
# --- Setup: code, packages, data -------------------------------------------------
# 1) Upload skyquery.zip as a Kaggle Dataset named "skyquery-code" and add it to this notebook.
# 2) Settings -> Accelerator: GPU T4 x2, Internet: on.
# 3) Optional (OWLv2 / Grounding DINO): Add-ons -> Secrets -> HF_TOKEN (read-only token).
import os, subprocess, sys, glob, shutil
CODE = glob.glob("/kaggle/input/skyquery-code/**/pyproject.toml", recursive=True)
assert CODE, "Add the skyquery-code dataset to this notebook"
SRC = os.path.dirname(CODE[0])
WORK = "/kaggle/working/skyquery"
if not os.path.exists(WORK):
    shutil.copytree(SRC, WORK, ignore=shutil.ignore_patterns("node_modules", "data", "cache"))
os.chdir(WORK)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[eval]", "transformers>=4.57", "git+https://github.com/ultralytics/CLIP.git"], check=True)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded")
except Exception:
    print("No HF_TOKEN secret: OWLv2 / Grounding DINO will try anonymous downloads")

os.makedirs("data", exist_ok=True); os.makedirs("cache", exist_ok=True)
BASE = "https://github.com/ultralytics/assets/releases/download/v0.0.0/"
def fetch(split):
    d = f"data/VisDrone2019-DET-{split}"
    if not os.path.exists(d):
        subprocess.run(f"curl -sSL -o /tmp/{split}.zip {BASE}VisDrone2019-DET-{split}.zip && unzip -q /tmp/{split}.zip -d data && rm /tmp/{split}.zip", shell=True, check=True)
    return d
TRAIN, VAL, TEST = fetch("train"), fetch("val"), fetch("test-dev")
HOLD = ["bus", "bicycle", "awning-tricycle"]


In [ ]:
DS = "/kaggle/working/vd_heldout"
if not os.path.exists(f"{DS}/data.yaml"):
    subprocess.run([sys.executable, "-m", "skyquery.calibrate.tiles", "--src", TRAIN, "--val", VAL, "--out", DS, "--holdout", *HOLD], check=True)
print(open(f"{DS}/holdout.json").read())


In [ ]:
from ultralytics import YOLO
RUN = "/kaggle/working/runs/heldout7"
DONE = f"{RUN}/weights/TRAINING_DONE"
if not os.path.exists(DONE):
    last = f"{RUN}/weights/last.pt"
    if os.path.exists(last):
        try:
            YOLO(last).train(resume=True)          # session restarted mid-training
        except Exception as e:
            print("resume not possible:", e)       # e.g. already finished
    else:
        YOLO("yolo26s.pt").train(data=f"{DS}/data.yaml", epochs=40, patience=15, batch=32, imgsz=640, device="0,1", workers=4,
                                 project="/kaggle/working/runs", name="heldout7", exist_ok=True, seed=42, cos_lr=True,
                                 close_mosaic=10, amp=True, max_det=902, plots=True)
    open(DONE, "w").write("ok")


In [ ]:
BEST = f"{RUN}/weights/best.pt"
for tag, split in (("val", VAL), ("testdev", TEST)):
    subprocess.run([sys.executable, "-m", "skyquery.calibrate.cache_preds", "--split", split, "--scale", "1280", "--device", "0",
                    "--out", f"cache/{tag}_1280.npz"], check=True)                     # 10-class oracle (your weights)
    subprocess.run([sys.executable, "-m", "skyquery.calibrate.cache_preds", "--split", split, "--scale", "1280", "--device", "0",
                    "--weights", BEST, "--out", f"cache/{tag}_heldout7_1280.npz"], check=True)
    subprocess.run([sys.executable, "-m", "skyquery.calibrate.cache_ovd", "--split", split, "--model", "yoloe-26l", "--prompts", "names",
                    "--classes", *HOLD, "--device", "0", "--out", f"cache/{tag}_yoloe26l_heldout.npz"], check=True)
    subprocess.run([sys.executable, "-m", "skyquery.calibrate.cache_ovd", "--split", split, "--model", "yoloe-26l", "--prompts", "names",
                    "--device", "0", "--out", f"cache/{tag}_yoloe26l_names.npz"], check=True)


In [ ]:
subprocess.run([sys.executable, "-m", "skyquery.calibrate.ovd_eval", "e2", "--val", VAL, "--test", TEST, "--holdout", *HOLD], check=True)
shutil.copy(BEST, "/kaggle/working/yolo26s_heldout7.pt")
subprocess.run("zip -qr /kaggle/working/e2_results.zip reports -i 'reports/*.json'", shell=True)
print("Download e2_results.zip and yolo26s_heldout7.pt from the Output panel")
